<a href="https://colab.research.google.com/github/MarinaSSilva/Assistente-de-Voz/blob/main/Assistente_de_Voz.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
language = 'pt' # Define o idioma como Português

#1. Gravação do Áudio com Python🎤

In [ ]:
from IPython.display import display, Javascript
from google.colab import output
from base64 import b64decode

Record = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var Record = time => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  await sleep(time)
  recorder.onstop = async ()=>{
    blob = new Blob(chunks)
    text = await b2text(blob)
    resolve(text)
  }
  recorder.stop()
})
"""

def recorder(sec=5):
    # Exibe e executa o código JavaScript definido na variável 'Record'
    display(Javascript(Record))

    # Executa a função JS e recebe o resultado em milissegundos
    js_result = output.eval_js(f'Record({sec * 1000})')

    # Decodifica o áudio de base64 para binário
    audio = b64decode(js_result.split(',')[1])

    file_name = 'request_audio.wav'
    # Salva o arquivo no sistema de arquivos do Colab
    with open(file_name, "wb") as f:
        f.write(audio)

    return f'/content/{file_name}' # Retorna o caminho do arquivo

print("ouvindo...\n")

# Exemplo de uso para gravar e exibir o player de áudio
record_file = recorder(5)
display(Audio(record_file, autoplay=True))

ouvindo...



<IPython.core.display.Javascript object>

#2. Reconhecimento de Fala com Whisper 🧠

In [ ]:
!pip install openai-whisper # Instalação da biblioteca
import whisper

model = whisper.load_model("small") # Carregamento do modelo

# Transcrição do áudio gravado na etapa anterior
result = model.transcribe(record_file, fp16=False, language=language)
transcription = result["text"] # Recupera apenas o texto do resultado

print(transcription)

 Olá mundo!


#3. Integração com a API do ChatGPT 💭

In [ ]:
!pip install openai --upgrade # Garante que a biblioteca está na versão atual
from openai import OpenAI

# 1. Instancia o cliente (em vez de usar openai.api_key diretamente)
client = OpenAI (api_key="sua chave aqui!") # Definição da chave de API

# 2. Criação da interação com a nova sintaxe
# O modelo 'gpt-3.5-turbo' e a estrutura de 'messages' seguem o padrão das fontes [2, 3]
response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "user", "content": transcription} # 'transcription' vem da Etapa 2 [3]
    ]
)

# 3. Extração da resposta de forma atualizada
# Conforme as fontes, acessamos a primeira escolha (índice 0) para obter o conteúdo [4]
chatgpt_response = response.choices.message.content

print(chatgpt_response)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

#4. Síntese de Voz - Text-to-Speech (gTTS) 🔊

In [ ]:
# Definindo a variável manualmente para contornar o erro de cota da Etapa 3
chatgpt_response = "Olá! Eu sou o seu assistente de voz. O código da etapa quatro está funcionando corretamente."

!pip install gTTS # Instalação da biblioteca
from gtts import gTTS
from IPython.display import Audio

# Instancia o objeto gTTS com o texto da resposta e o idioma
tts = gTTS(text=chatgpt_response, lang=language, slow=False)

# Salva o arquivo de áudio resultante
response_audio_path = '/content/response_audio.mp3'
tts.save(response_audio_path)

# Exibe o player com reprodução automática
display(Audio(response_audio_path, autoplay=True))